# 00. 실습실 PC 환경 점검 (전원 공통)

담당: 팀원 전원이 각자의 실습실 Windows PC에서 가장 먼저 실행합니다.
목표: 설치를 끝내고, torch가 RTX 5060 Ti를 cu128 빌드로 인식하는지 확인하고, 가벼운 preflight를 통과시킨 뒤,
그 결과를 팀 채널에 붙여 넣을 수 있는 형태로 출력하는 것입니다.

소요 시간은 온라인 설치 기준으로 15분에서 30분 사이입니다. 오프라인 설치라면 5분 내외로 끝납니다.

## 통과 기준(PASS)
1. `torch.__version__` 에 `+cu128` 이 포함되고 `torch.cuda.is_available()` 이 True입니다.
2. `torch.cuda.get_device_name()` 이 RTX 5060 Ti를 가리키고, compute capability가 `(12, 0)` 입니다.
3. `python -m vqa.preflight` 가 종료 코드 0으로 끝나고 결과 JSON이 저장됩니다.

## 넘겨줄 내용
마지막 셀이 출력하는 결과 블록을 팀 채널에 그대로 붙여 넣습니다. 준혁과 유림이 이 값을 보고
각 PC에 어떤 작업을 배정할지 정합니다.

## 1. 파라미터

아래 셀만 자신의 PC에 맞게 고치면 됩니다.

- `INSTALL_MODE`: 인터넷이 되면 `"online"`, 안 되면 `"offline"` 으로 둡니다.
- `TF_TRACK`: `"tf4"` 는 transformers 4.x 환경이며 Qwen2.5-VL-3B로 오늘 바로 돌릴 수 있습니다.
  `"tf5"` 는 transformers 5.x 환경이며 Qwen3.5 계열을 돌릴 때 필요합니다. 두 환경은 conda 가상환경을
  따로 만들어서 병행하는 것을 권장합니다. 자세한 절차는 `docs/lab_setup.md` 를 보십시오.

In [ ]:
# ===== 파라미터 =====
PROJECT_ROOT = r"."              # 이 저장소의 위치. 예: r"C:\vqa\ssafy-16-2-ai"
DATA_ROOT = r"."                 # train.csv, train/, test/ 가 들어 있는 폴더
RUN_ROOT = r"./artifacts/runs"   # 산출물이 쌓이는 영속 폴더
LOCAL_SCRATCH = r"./artifacts/scratch"   # 빠른 로컬 디스크
MODEL_ROOT = r""                 # 오프라인 모델 폴더나 HF 캐시. 없으면 빈 문자열로 둡니다.

TEAM_MEMBER = "이름을적으세요"    # 예: "수빈"
PC_LABEL = "lab-pc-1"            # 실습실 PC를 구분할 이름

INSTALL_MODE = "online"          # "online" 또는 "offline"
TF_TRACK = "tf4"                 # "tf4" 또는 "tf5"
WHEEL_DIR = r"./downloads/libs"  # 오프라인 설치에 쓸 wheel 폴더

PREFLIGHT_TRAIN_STEPS = 0        # 가벼운 점검이므로 학습 step은 생략합니다.
PREFLIGHT_INFER_ITEMS = 5        # 추론은 5문항만 확인합니다.

In [ ]:
# --- 공통 부트스트랩: 경로, 환경 변수, 명령 실행 헬퍼 (수정할 일 거의 없음) ---
import os, sys, json, shlex, subprocess, time, csv, hashlib, random
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def _setenv(name, value, create=False):
    if not value:
        return None
    p = Path(value).expanduser()
    if create:
        p.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(p)
    return p


DATA_ROOT_P = _setenv("VQA_DATA_ROOT", DATA_ROOT)
RUN_ROOT_P = _setenv("VQA_RUN_ROOT", RUN_ROOT, create=True)
SCRATCH_P = _setenv("VQA_LOCAL_SCRATCH", LOCAL_SCRATCH, create=True)
MODEL_ROOT_P = _setenv("VQA_MODEL_ROOT", MODEL_ROOT)

PY = sys.executable


def run(cmd, check=True, echo=True, quiet=False, hint=True):
    """명령을 실행하면서 출력을 그대로 보여 주고 종료 코드와 소요 시간을 돌려줍니다.

    quiet가 True이면 명령의 출력도, 뒤에 붙는 "[rc=...]" 줄도 찍지 않습니다. 상태를 조회하는
    호출은 실패가 정상적인 결과입니다. 예를 들어 `vqa.jobs status` 는 죽은 job에 대해 종료 코드
    1을 돌려줍니다. 그 줄을 그대로 찍으면 셀 출력 맨 앞에 "[rc=1]"이 남아서, 실제 실패로 읽히는
    오해를 부릅니다.
    """
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    cmd = [str(x) for x in cmd]
    if echo:
        print("$ " + " ".join(shlex.quote(x) for x in cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        cmd, cwd=str(PROJECT_ROOT), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        errors="replace", bufsize=1,
    )
    buf = []
    for line in proc.stdout:
        buf.append(line)
        if not quiet:
            print(line, end="", flush=True)
    rc = proc.wait()
    dt = time.time() - t0
    if not quiet:
        print("[rc=%d] %.1f초" % (rc, dt), flush=True)
    if rc != 0:
        if hint:
            print("실패했습니다. 같은 명령에 --help 를 붙여서 인자 이름부터 확인하십시오.", flush=True)
        if check:
            raise RuntimeError("명령이 실패했습니다: " + " ".join(cmd))
    return {"rc": rc, "seconds": dt, "output": "".join(buf)}


def vqa(module, *args, **kw):
    """python -m vqa.<module> 을 실행합니다."""
    return run([PY, "-m", "vqa." + module] + [str(a) for a in args], **kw)


def script(name, *args, **kw):
    """scripts/<name> 을 실행합니다. name 에 / 가 있으면 저장소 루트 기준 경로로 봅니다."""
    path = Path(name) if "/" in name else Path("scripts") / name
    return run([PY, str(path)] + [str(a) for a in args], **kw)


def read_csv_rows(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def read_ids(path):
    return [ln.strip() for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip()]


def sha256_of(path, limit=None):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    return digest[:limit] if limit else digest


print("PROJECT_ROOT      :", PROJECT_ROOT)
for _name in ("VQA_DATA_ROOT", "VQA_RUN_ROOT", "VQA_LOCAL_SCRATCH", "VQA_MODEL_ROOT"):
    print("%-18s: %s" % (_name, os.environ.get(_name, "(미설정)")))

## 2. 라이브러리 설치

`INSTALL_MODE` 와 `TF_TRACK` 조합에 따라 설치 명령이 정해집니다. 오프라인이면 `--no-index` 를 붙여서
`downloads/libs` 의 wheel만 사용하므로 인터넷에 접속하지 않습니다.

설치가 끝난 뒤에 torch를 처음 import 하는 경우에는 커널을 한 번 재시작하는 편이 안전합니다.
재시작한 뒤에는 파라미터 셀과 부트스트랩 셀부터 다시 실행하십시오.

In [ ]:
REQ_FILE = {"tf4": "requirements/lab_windows_tf4.txt",
            "tf5": "requirements/lab_windows_tf5.txt"}[TF_TRACK]

if not Path(REQ_FILE).exists():
    raise FileNotFoundError(
        REQ_FILE + " 파일이 없습니다. 코어 담당자가 requirements/ 를 올렸는지 확인하십시오."
    )

pip_cmd = [PY, "-m", "pip", "install", "-r", REQ_FILE]
if INSTALL_MODE == "offline":
    wheel_dir = Path(WHEEL_DIR).expanduser()
    if not wheel_dir.is_dir():
        raise FileNotFoundError(str(wheel_dir) + " 폴더가 없습니다. 오프라인 wheel을 먼저 준비하십시오.")
    pip_cmd += ["--no-index", "--find-links", str(wheel_dir)]
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    print("오프라인 모드로 설치합니다. 인터넷에 접속하지 않습니다.")
else:
    print("온라인 모드로 설치합니다.")

run(pip_cmd)

## 3. torch가 GPU를 제대로 인식하는지 확인

별도 프로세스에서 확인하므로, torch import 단계에서 문제가 생겨도 노트북 커널은 살아 있습니다.
`+cu128` 이 아닌 CPU 전용 torch가 잡히면 실패로 처리합니다.

In [ ]:
PROBE = r"""
import json, platform, sys
info = {"python": sys.version.split()[0], "platform": platform.platform()}
try:
    import torch
    info["torch"] = torch.__version__
    info["torch_cuda"] = torch.version.cuda
    info["cuda_available"] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        info["device_name"] = torch.cuda.get_device_name(0)
        info["capability"] = list(torch.cuda.get_device_capability(0))
        info["vram_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 1)
except Exception as exc:
    info["torch_error"] = repr(exc)
for mod in ("transformers", "peft", "accelerate", "bitsandbytes"):
    try:
        info[mod] = __import__(mod).__version__
    except Exception as exc:
        info[mod] = "미설치 또는 오류: " + repr(exc)[:80]
print("VQA_ENV_JSON " + json.dumps(info, ensure_ascii=False))
"""

res = run([PY, "-c", PROBE], check=False, quiet=True)
ENV_INFO = {}
for line in res["output"].splitlines():
    if line.startswith("VQA_ENV_JSON "):
        ENV_INFO = json.loads(line[len("VQA_ENV_JSON "):])
    else:
        print(line)

for k, v in ENV_INFO.items():
    print("%-16s: %s" % (k, v))

problems = []
if "+cu" not in str(ENV_INFO.get("torch", "")):
    problems.append("torch가 CPU 전용 빌드입니다. cu128 wheel로 다시 설치해야 합니다.")
if "cu128" not in str(ENV_INFO.get("torch", "")):
    problems.append("torch 빌드 태그에 cu128이 없습니다. 실습실 표준과 다른 빌드일 수 있습니다.")
if not ENV_INFO.get("cuda_available"):
    problems.append("torch.cuda.is_available()가 False입니다. 드라이버와 설치를 확인하십시오.")
if ENV_INFO.get("capability") and list(ENV_INFO["capability"]) != [12, 0]:
    problems.append("compute capability가 (12, 0)이 아닙니다. 실제 값: %s" % (ENV_INFO["capability"],))

if problems:
    print()
    print("확인이 필요한 항목")
    for p in problems:
        print(" - " + p)
else:
    print()
    print("PASS: torch가 Blackwell GPU를 cu128 빌드로 인식했습니다.")

## 4. 가벼운 preflight

학습 step은 0으로 두고 추론만 몇 문항 확인합니다. 본격적인 20-step 점검은 학습 담당인 유림이
`12_lab_train_9b_qlora.ipynb` 에서 수행하므로, 여기에서 반복할 필요는 없습니다.

`tf4` 환경이면 Qwen2.5-VL-3B config를, `tf5` 환경이면 Qwen3.5-9B config를 사용합니다.

In [ ]:
CONFIG = {"tf4": "configs/lab_qwen25vl_3b.yaml",
          "tf5": "configs/lab_qwen35_9b.yaml"}[TF_TRACK]

print("사용할 config:", CONFIG)
if not Path(CONFIG).exists():
    print("아직 config 파일이 없습니다. 코어 담당자의 커밋을 기다렸다가 다시 실행하십시오.")
else:
    preflight_res = vqa(
        "preflight",
        "--config", CONFIG,
        "--n-train-steps", PREFLIGHT_TRAIN_STEPS,
        "--n-infer", PREFLIGHT_INFER_ITEMS,
        check=False,
    )
    print("preflight 종료 코드:", preflight_res["rc"])

## 5. 팀 채널에 공유할 결과 블록

아래 셀의 출력을 그대로 복사해서 팀 채널에 붙여 넣으십시오. 준혁이 이 값을 모아서 PC 배정과
시간 예산을 계산합니다.

In [ ]:
latest_preflight = None
pf_dir = RUN_ROOT_P / "preflight" if RUN_ROOT_P else None
if pf_dir and pf_dir.is_dir():
    files = sorted(pf_dir.glob("*.json"))
    if files:
        latest_preflight = files[-1]

lines = []
lines.append("===== 실습실 환경 점검 결과 =====")
lines.append("담당자      : %s" % TEAM_MEMBER)
lines.append("PC          : %s" % PC_LABEL)
lines.append("설치 모드   : %s / transformers 계열 %s" % (INSTALL_MODE, TF_TRACK))
lines.append("python      : %s" % ENV_INFO.get("python"))
lines.append("torch       : %s (cuda %s)" % (ENV_INFO.get("torch"), ENV_INFO.get("torch_cuda")))
lines.append("GPU         : %s / VRAM %s GB / capability %s"
             % (ENV_INFO.get("device_name"), ENV_INFO.get("vram_gb"), ENV_INFO.get("capability")))
lines.append("transformers: %s" % ENV_INFO.get("transformers"))
lines.append("peft        : %s" % ENV_INFO.get("peft"))
lines.append("bitsandbytes: %s" % ENV_INFO.get("bitsandbytes"))
lines.append("preflight   : %s" % (latest_preflight.name if latest_preflight else "아직 없음"))
lines.append("판정        : %s" % ("PASS" if not problems else "확인 필요 %d건" % len(problems)))
for p in problems:
    lines.append("  - %s" % p)
lines.append("================================")

print("\n".join(lines))